In [1]:
!pip install pandas sqlalchemy oracledb


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('D:/CSDLPT/new_dataset.csv')

In [4]:
df.head()

,order_id,order_date,order_status,customer_id,customer_name,gender,age,customer_segment,country,city,...,brand,stock_quantity,unit_price_usd,quantity,discount_percent,discount_amount_usd,total_price_usd,tax_usd,payment_method,warehouse_location
0,ORD-XAJI0,2024-10-14 11:20:05.496679,Completed,CUS-6DPBH,Drew Warren,Male,53,Regular,Netherlands,Utrecht,...,Nike,99,120.51,3,10,36.15,325.38,28.14,Apple Pay,UK
1,ORD-NHJ7X,2024-10-21 00:49:44.681065,Completed,CUS-G0FN9,Victor Sullivan,Male,61,Regular,United States,Los Angeles,...,Asus,146,60.33,2,15,18.10,102.56,8.96,Bank Transfer,UK
2,ORD-YTJXE,2025-03-17 19:49:36.983317,Completed,CUS-Q85JS,Jason Garcia,Male,50,VIP,France,Nice,...,Asus,899,295.89,3,0,0.00,887.67,69.56,Credit Card,UK
3,ORD-EIMVI,2024-09-27 06:24:44.913768,Completed,CUS-CWI64,David Norris,Male,50,Regular,Italy,Milan,...,Adidas,884,186.33,1,0,0.00,186.33,16.78,Credit Card,USA-West
4,ORD-OR56F,2025-05-21 17:10:48.401882,Completed,CUS-O72KZ,Laura Phillips,Female,27,Regular,Germany,Berlin,...,Anker,82,427.16,1,10,42.72,384.44,23.74,Credit Card,UK


In [5]:
# Lọc các cột cho bảng CUSTOMERS và loại bỏ trùng lặp customer_id
customers_df = df[[
    'customer_name', 
    'gender', 
    'age', 
    'country', 
    'city'
]].drop_duplicates()

In [6]:
from sqlalchemy import create_engine

In [7]:
# --- BƯỚC 2: CẤU HÌNH KẾT NỐI ORACLE LOCAL ---
#user connection 
USER = 'branch3'      
PASSWORD = 'branch3'
HOST = 'localhost'
PORT = '1521'
SERVICE_NAME = 'orcl'         # Thường là xe hoặc orcl

# Tạo connection string cho SQLAlchemy
# Sử dụng 'oracle+oracledb' để dùng thư viện thế hệ mới
conn_url = f"oracle+oracledb://{USER}:{PASSWORD}@{HOST}:{PORT}/?service_name={SERVICE_NAME}"
engine = create_engine(conn_url)

In [8]:
# --- BƯỚC 3: NẠP DỮ LIỆU ---
try:
    print("Đang nạp dữ liệu vào bảng CUSTOMERS...")
    customers_df.to_sql(
        name='CUSTOMERS', 
        con=engine, 
        if_exists='append', 
        index=False
    )
    print("Thành công! Dữ liệu đã được nạp vào Oracle Local.")
except Exception as e:
    print(f"Lỗi: {e}")

Đang nạp dữ liệu vào bảng CUSTOMERS...
Thành công! Dữ liệu đã được nạp vào Oracle Local.


C:\Users\meiln\AppData\Local\Temp\ipykernel_4232\471193883.py:4: UserWarning: The provided table name 'CUSTOMERS' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  customers_df.to_sql(


In [9]:
# 1. Lọc và làm sạch dữ liệu cho bảng PRODUCTS_INFO
# Dựa trên danh sách cột bạn đã liệt kê ở yêu cầu trước
products_df = df[[
    'product_name', 
    'category', 
    'brand'
]].drop_duplicates()  # Loại bỏ các sản phẩm trùng tên, loại, nhãn hiệu

# 2. Đẩy dữ liệu vào bảng trong Oracle
try:
    print("Đang nạp dữ liệu vào bảng PRODUCTS_INFO...")
    
    products_df.to_sql(
        name='PRODUCTS_INFO', 
        con=engine, 
        if_exists='append', # Dùng 'append' để thêm dữ liệu vào bảng đã có
        index=False
    )
    
    print(f"✅ Thành công! Đã nạp {len(products_df)} sản phẩm vào PRODUCTS_INFO.")
    
except Exception as e:
    print(f"❌ Lỗi khi nạp bảng PRODUCTS_INFO: {e}")

Đang nạp dữ liệu vào bảng PRODUCTS_INFO...
✅ Thành công! Đã nạp 960 sản phẩm vào PRODUCTS_INFO.


C:\Users\meiln\AppData\Local\Temp\ipykernel_4232\2434615205.py:13: UserWarning: The provided table name 'PRODUCTS_INFO' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  products_df.to_sql(


In [10]:
import pandas as pd
import numpy as np

if len(df) < 960:
    source_df = pd.concat([df] * (960 // len(df) + 1), ignore_index=True)
else:
    source_df = df.copy()

# 2. Chỉ giữ lại 960 dòng đầu tiên và các cột cần thiết
stock_df = source_df.iloc[:960][['stock_quantity', 'unit_price_usd']].copy()

# 3. Tạo product_id từ 1 đến 960
stock_df['product_id'] = range(1, 961)

# 4. Gán branch_id là id chi nhánh của máy
stock_df['branch_id'] = 3


stock_df['stock_quantity'] = stock_df['stock_quantity'].apply(
    lambda x: x if pd.notnull(x) else np.random.randint(10, 501)
)

stock_df['unit_price_usd'] = stock_df['unit_price_usd'].apply(
    lambda x: x if pd.notnull(x) else round(np.random.uniform(5, 1000), 2)
)

stock_df = stock_df[['product_id', 'branch_id', 'stock_quantity', 'unit_price_usd']]

# 7. Nạp vào Oracle
from sqlalchemy import types

# kiểm tra đang login đúng user/schema chưa
with engine.connect() as conn:
    print(conn.exec_driver_sql("""
        SELECT USER, SYS_CONTEXT('USERENV','CURRENT_SCHEMA') 
        FROM dual
    """).fetchone())

stock_df['product_id'] = stock_df['product_id'].astype(int)
stock_df['branch_id'] = stock_df['branch_id'].astype(int)
stock_df['stock_quantity'] = stock_df['stock_quantity'].astype(int)
stock_df['unit_price_usd'] = stock_df['unit_price_usd'].astype(float)

try:
    print("Đang nạp dữ liệu vào PRODUCTS_STOCK cho chi nhánh 2...")

    stock_df.to_sql(
        name='PRODUCTS_STOCK',
        con=engine,
        schema='branch3',
        if_exists='append',
        index=False,
        chunksize=200,
        dtype={
            'product_id': types.Integer(),
            'branch_id': types.Integer(),
            'stock_quantity': types.Integer(),
            'unit_price_usd': types.Numeric(10, 2)
        }
    )

    print(f"✅ Thành công! Đã nạp {len(stock_df)} dòng vào PRODUCTS_STOCK.")
except Exception as e:
    print(f"❌ Lỗi: {e}")

('BRANCH3', 'BRANCH3')
Đang nạp dữ liệu vào PRODUCTS_STOCK cho chi nhánh 2...
✅ Thành công! Đã nạp 960 dòng vào PRODUCTS_STOCK.


C:\Users\meiln\AppData\Local\Temp\ipykernel_4232\105299335.py:47: UserWarning: The provided table name 'PRODUCTS_STOCK' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  stock_df.to_sql(


insert employee

In [11]:
df2 = pd.read_csv('employees_350_rows.csv')
df2['hire_date'] = pd.to_datetime(df2['hire_date'])

In [12]:
df2.head()

,employee_id,first_name,last_name,email,phone_number,hire_date,salary,branch_id
0,701,Jeffrey,Lewis,jeffrey.lewis701@company.com,166857903,2019-06-19,10803.45,1
1,702,Jason,Jackson,jason.jackson702@company.com,902545040,2024-04-16,3819.73,2
2,703,Thomas,Taylor,thomas.taylor703@company.com,692560683,2023-10-04,4358.10,1
3,704,Andrew,Brown,andrew.brown704@company.com,117198750,2025-05-31,6754.98,1
4,705,James,Wilson,james.wilson705@company.com,602919245,2020-01-27,3779.17,2


In [13]:
# Xóa khoảng trắng thừa ở đầu và cuối tên cột
df2.columns = df2.columns.str.strip()

# Bây giờ chạy lại dòng này sẽ hết lỗi
df2['phone_number'] = '0' + df2['phone_number'].astype(str)

In [14]:
from sqlalchemy import create_engine, types

In [15]:
from sqlalchemy import types

# 1. Loại bỏ cột thừa 'Unnamed: 0' phát sinh từ file CSV (nếu có)
if 'Unnamed: 0' in df2.columns:
    df2 = df2.drop(columns=['Unnamed: 0'])

# 2. Tiến hành nạp vào Oracle
print("Đang nạp dữ liệu vào bảng EMPLOYEES...")
try:
    df2.to_sql(
        name='EMPLOYEES',
        con=engine,
        if_exists='append',
        index=False,
        dtype={
            'hire_date': types.Date(),
            'phone_number': types.VARCHAR(20),
            'salary': types.Numeric() # 🌟 Thêm dòng này để né lỗi FLOAT nhị phân của Oracle
        }
    )
    print(f"✅ Thành công! Đã nạp {len(df2)} nhân viên vào hệ thống.")
except Exception as e:
    print(f"❌ Lỗi khi nạp bảng: {e}")

Đang nạp dữ liệu vào bảng EMPLOYEES...
✅ Thành công! Đã nạp 350 nhân viên vào hệ thống.


C:\Users\meiln\AppData\Local\Temp\ipykernel_4232\3207591414.py:10: UserWarning: The provided table name 'EMPLOYEES' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df2.to_sql(


table order

In [62]:
# 3. Đọc dữ liệu từ file CSV gốc
df_orders_raw = pd.read_csv('D:/CSDLPT/Dataset/df_part_3.csv')
df_orders_raw.columns = df_orders_raw.columns.str.strip() # Xóa khoảng trắng tên cột


In [17]:
df_orders_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 333375 entries, 0 to 333374
Data columns (total 25 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   order_id                  333375 non-null  str    
 1   order_date                333375 non-null  str    
 2   order_status              333375 non-null  str    
 3   customer_id               333375 non-null  str    
 4   customer_name             333375 non-null  str    
 5   gender                    333375 non-null  str    
 6   age                       333375 non-null  int64  
 7   customer_segment          333375 non-null  str    
 8   country                   333375 non-null  str    
 9   city                      333375 non-null  str    
 10  customer_loyalty_score    333375 non-null  float64
 11  total_orders_by_customer  333375 non-null  int64  
 12  product_id                333375 non-null  str    
 13  product_name              333375 non-null  str    
 14 

In [18]:
df_orders_raw.head()

,order_id,order_date,order_status,customer_id,customer_name,gender,age,customer_segment,country,city,...,brand,stock_quantity,unit_price_usd,quantity,discount_percent,discount_amount_usd,total_price_usd,tax_usd,payment_method,warehouse_location
0,ORD-8VFLV,2025-09-24 22:44:05.262791,Completed,CUS-08WDW,Tony Medina,Male,38,Regular,Italy,Rome,...,Adidas,391,449.68,5,0,0.00,2248.40,158.56,Debit Card,USA-East
1,ORD-PKS0P,2024-06-03 12:16:05.367571,Completed,CUS-FLUVN,Madeline Morrison,Female,32,VIP,Australia,Perth,...,JBL,914,219.26,2,10,43.85,394.67,30.75,Debit Card,USA-West
2,ORD-8LZJB,2025-03-02 06:36:11.910794,Completed,CUS-VBUIW,James Hunter,Male,72,Regular,France,Lyon,...,Baseus,364,472.29,5,0,0.00,2361.45,181.87,Bank Transfer,USA-East
3,ORD-QR558,2025-11-18 19:27:53.044985,Completed,CUS-IG10U,Heather Pruitt,Female,66,Regular,United States,New York,...,Logitech,791,56.40,3,0,0.00,169.20,10.67,PayPal,UK
4,ORD-CGL8K,2025-08-03 08:50:52.434477,Completed,CUS-P65RV,Paul Carroll,Male,51,VIP,Netherlands,Eindhoven,...,Samsung,450,388.71,1,10,38.87,349.84,17.75,Debit Card,EU-Central


In [19]:
# 1. Lấy dữ liệu tra cứu từ Oracle (Lấy thêm Age và Gender để khớp)
cust_lookup = pd.read_sql("SELECT customer_id, customer_name, age, gender FROM CUSTOMERS", engine)
cust_lookup.columns = cust_lookup.columns.str.lower()
# Làm sạch dữ liệu tra cứu
cust_lookup['customer_name'] = cust_lookup['customer_name'].astype(str).str.strip()
cust_lookup['gender'] = cust_lookup['gender'].astype(str).str.strip()
cust_lookup['age'] = cust_lookup['age'].astype(float)

In [20]:
df_orders_raw.columns = df_orders_raw.columns.str.strip().str.lower()
df_orders_raw['customer_name'] = df_orders_raw['customer_name'].astype(str).str.strip()
df_orders_raw['gender'] = df_orders_raw['gender'].astype(str).str.strip()
df_orders_raw['age'] = df_orders_raw['age'].astype(float)

In [21]:
# 3. Thực hiện Merge trên NHIỀU CỘT để đảm bảo duy nhất
# Việc map thêm age và gender sẽ giúp loại bỏ trường hợp trùng tên nhưng khác người
df_orders = pd.merge(
    df_orders_raw, 
    cust_lookup, 
    on=['customer_name', 'age', 'gender'], 
    how='inner'
)

In [22]:
# KIỂM TRA: Nếu merge xong mà không có dữ liệu, báo lỗi ngay
if df_orders.empty:
    print("❌ LỖI: Không tìm thấy khách hàng nào khớp giữa CSV và Database!")
    print("Vui lòng kiểm tra lại cột 'customer_name' ở cả 2 nơi.")
else:
    print("OK")

OK


In [23]:
df_orders.drop_duplicates(['order_id', 'order_date', 'product_id'],inplace=True)

In [24]:
df_orders.head(20)

,order_id,order_date,order_status,customer_id_x,customer_name,gender,age,customer_segment,country,city,...,stock_quantity,unit_price_usd,quantity,discount_percent,discount_amount_usd,total_price_usd,tax_usd,payment_method,warehouse_location,customer_id_y
0,ORD-8VFLV,2025-09-24 22:44:05.262791,Completed,CUS-08WDW,Tony Medina,Male,38.0,Regular,Italy,Rome,...,391,449.68,5,0,0.00,2248.40,158.56,Debit Card,USA-East,665594
1,ORD-PKS0P,2024-06-03 12:16:05.367571,Completed,CUS-FLUVN,Madeline Morrison,Female,32.0,VIP,Australia,Perth,...,914,219.26,2,10,43.85,394.67,30.75,Debit Card,USA-West,665595
2,ORD-8LZJB,2025-03-02 06:36:11.910794,Completed,CUS-VBUIW,James Hunter,Male,72.0,Regular,France,Lyon,...,364,472.29,5,0,0.00,2361.45,181.87,Bank Transfer,USA-East,541841
4,ORD-QR558,2025-11-18 19:27:53.044985,Completed,CUS-IG10U,Heather Pruitt,Female,66.0,Regular,United States,New York,...,791,56.40,3,0,0.00,169.20,10.67,PayPal,UK,665597
5,ORD-CGL8K,2025-08-03 08:50:52.434477,Completed,CUS-P65RV,Paul Carroll,Male,51.0,VIP,Netherlands,Eindhoven,...,450,388.71,1,10,38.87,349.84,17.75,Debit Card,EU-Central,593341
7,ORD-IEX0W,2025-06-29 09:50:18.289380,Completed,CUS-1MSG5,Brandon Morgan,Male,73.0,Regular,United Kingdom,London,...,857,148.52,2,10,29.70,267.34,13.66,Debit Card,USA-West,665599
8,ORD-0EZNX,2025-04-02 19:28:25.389409,Completed,CUS-LDXTR,Carlos Johnson,Male,60.0,Regular,Canada,Calgary,...,824,129.00,2,0,0.00,258.00,25.09,Credit Card,Asia-Pacific,665600
9,ORD-CJ5PF,2025-08-17 22:57:01.879178,Completed,CUS-IZ74A,Anthony Mcclure,Male,23.0,Premium,Spain,Valencia,...,739,99.68,4,0,0.00,398.72,26.39,PayPal,USA-West,665601
10,ORD-BRJAR,2025-05-31 10:58:37.685578,Completed,CUS-J9EXF,Sara Carrillo,Female,26.0,Premium,United Kingdom,Glasgow,...,556,346.94,2,25,173.47,520.41,28.09,Apple Pay,USA-West,665602
11,ORD-6WC14,2025-01-12 21:08:42.439154,Completed,CUS-SMVZ3,George Stewart,Male,56.0,Regular,Netherlands,Eindhoven,...,281,127.15,1,20,25.43,101.72,5.94,Credit Card,USA-West,665603


In [25]:
# 4. Kiểm tra số lượng dòng sau khi merge
print(f"Số dòng ban đầu: {len(df_orders_raw)}")
print(f"Số dòng sau khi merge đa điều kiện: {len(df_orders)}")

Số dòng ban đầu: 333375
Số dòng sau khi merge đa điều kiện: 333375


In [26]:
# Gán các giá trị mặc định
df_orders['branch_id'] = 3

# Giả sử ID nhân viên của bạn từ 100 đến 200 
df_orders['employee_id'] = np.random.randint(701, 1050, size=len(df_orders))

# Chuyển đổi ngày tháng (đảm bảo đúng cột 'order_date')
if 'order_date' in df_orders.columns:
    df_orders['order_date'] = pd.to_datetime(df_orders['order_date'])
    df_orders = df_orders.sort_values(by='order_date').reset_index(drop=True)

In [27]:
if 'customer_id_y' in df_orders.columns:
    df_orders = df_orders.rename(columns={'customer_id_y': 'customer_id'})

In [28]:
df_orders.head()

,order_id,order_date,order_status,customer_id_x,customer_name,gender,age,customer_segment,country,city,...,quantity,discount_percent,discount_amount_usd,total_price_usd,tax_usd,payment_method,warehouse_location,customer_id,branch_id,employee_id
0,ORD-A9GBX,2024-02-03 04:34:14.421205,Pending,CUS-0WIYC,Mrs. Cassandra Parker,Female,66.0,Regular,Germany,Frankfurt,...,1,15,34.79,197.17,18.69,Debit Card,EU-Central,835508,3,1006
1,ORD-N36GF,2024-02-03 04:44:59.875980,Completed,CUS-UHXHM,Blake Jones,Male,67.0,Regular,Germany,Frankfurt,...,3,5,3.64,69.23,6.46,Apple Pay,EU-Central,858942,3,750
2,ORD-JPXWG,2024-02-03 04:45:57.159855,Completed,CUS-JDNIW,Anna Farley,Female,33.0,Premium,Netherlands,Utrecht,...,3,20,41.28,165.12,10.98,PayPal,USA-West,850889,3,706
3,ORD-8QWEJ,2024-02-03 04:49:13.724187,Completed,CUS-MTNBH,Stacey Williams,Female,20.0,Premium,Belgium,Bruges,...,1,5,7.89,149.91,12.45,Debit Card,EU-Central,547926,3,964
4,ORD-G47QE,2024-02-03 04:49:18.970409,Completed,CUS-9E21N,William Brooks,Male,29.0,Regular,United States,Houston,...,5,15,70.42,399.03,32.65,Credit Card,UK,696278,3,794


In [29]:
# Chỉ lấy các cột cần thiết cho bảng ORDERS
# Lưu ý: Cột ID từ cust_lookup chắc chắn là 'customer_id'tự thêm yếu số phân biệt .lower() ở trên
cols_to_keep = [
    'order_date', 'customer_id', 'employee_id', 
    'branch_id', 'payment_method', 'tax_usd', 'total_price_usd'
]

In [30]:
df_orders[cols_to_keep].info()

<class 'pandas.DataFrame'>
RangeIndex: 333375 entries, 0 to 333374
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   order_date       333375 non-null  datetime64[us]
 1   customer_id      333375 non-null  int64         
 2   employee_id      333375 non-null  int32         
 3   branch_id        333375 non-null  int64         
 4   payment_method   333375 non-null  str           
 5   tax_usd          333375 non-null  float64       
 6   total_price_usd  333375 non-null  float64       
dtypes: datetime64[us](1), float64(2), int32(1), int64(2), str(1)
memory usage: 16.5 MB


In [31]:
final_orders = df_orders[cols_to_keep]

In [57]:
# Nạp vào Oracle
final_orders.to_sql('ORDERS', con=engine, if_exists='append', index=False)
print(f"✅ Thành công! Đã nạp {len(final_orders)} đơn hàng.")


✅ Thành công! Đã nạp 333375 đơn hàng.


C:\Users\meiln\AppData\Local\Temp\ipykernel_4232\1215381642.py:2: UserWarning: The provided table name 'ORDERS' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  final_orders.to_sql('ORDERS', con=engine, if_exists='append', index=False)


insert order_items

In [60]:
import pandas as pd

# 1. Lấy bảng tra cứu từ Database
# Cần Order_id (mới sinh) và một mã tham chiếu
orders_db = pd.read_sql("SELECT order_id, order_date, customer_id FROM ORDERS", engine)
# Cần Product_id và giá bán
# Lấy Product_id và giá bán tại chi nhánh 2
query = """
    SELECT pi.product_id, pi.product_name, ps.unit_price_usd 
    FROM PRODUCTS_INFO pi
    JOIN PRODUCTS_STOCK ps ON pi.product_id = ps.product_id
    WHERE ps.branch_id = 3
"""
products_db = pd.read_sql(query, engine)
products_db.columns = products_db.columns.str.lower()

In [63]:
# 3. MAPPING (Đây là bước quan trọng nhất)
# Bước A: Map để lấy Product_i
df_items = pd.merge(df_orders_raw, products_db, on='product_name', how='inner')


In [64]:
df_items.drop_duplicates(['order_id', 'order_date', 'product_id_x'],inplace=True)

In [65]:
df_items.info() 

<class 'pandas.DataFrame'>
RangeIndex: 333375 entries, 0 to 6667480
Data columns (total 27 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   order_id                  333375 non-null  str    
 1   order_date                333375 non-null  str    
 2   order_status              333375 non-null  str    
 3   customer_id               333375 non-null  str    
 4   customer_name             333375 non-null  str    
 5   gender                    333375 non-null  str    
 6   age                       333375 non-null  int64  
 7   customer_segment          333375 non-null  str    
 8   country                   333375 non-null  str    
 9   city                      333375 non-null  str    
 10  customer_loyalty_score    333375 non-null  float64
 11  total_orders_by_customer  333375 non-null  int64  
 12  product_id_x              333375 non-null  str    
 13  product_name              333375 non-null  str    
 14

In [66]:
df_items.head()

,order_id,order_date,order_status,customer_id,customer_name,gender,age,customer_segment,country,city,...,unit_price_usd_x,quantity,discount_percent,discount_amount_usd,total_price_usd,tax_usd,payment_method,warehouse_location,product_id_y,unit_price_usd_y
0,ORD-8VFLV,2025-09-24 22:44:05.262791,Completed,CUS-08WDW,Tony Medina,Male,38,Regular,Italy,Rome,...,449.68,5,0,0.00,2248.40,158.56,Debit Card,USA-East,691,87.86
20,ORD-PKS0P,2024-06-03 12:16:05.367571,Completed,CUS-FLUVN,Madeline Morrison,Female,32,VIP,Australia,Perth,...,219.26,2,10,43.85,394.67,30.75,Debit Card,USA-West,683,118.09
40,ORD-8LZJB,2025-03-02 06:36:11.910794,Completed,CUS-VBUIW,James Hunter,Male,72,Regular,France,Lyon,...,472.29,5,0,0.00,2361.45,181.87,Bank Transfer,USA-East,675,409.15
60,ORD-QR558,2025-11-18 19:27:53.044985,Completed,CUS-IG10U,Heather Pruitt,Female,66,Regular,United States,New York,...,56.40,3,0,0.00,169.20,10.67,PayPal,UK,779,454.20
80,ORD-CGL8K,2025-08-03 08:50:52.434477,Completed,CUS-P65RV,Paul Carroll,Male,51,VIP,Netherlands,Eindhoven,...,388.71,1,10,38.87,349.84,17.75,Debit Card,EU-Central,744,381.85


In [67]:
# 1. Truy vấn lấy bảng khách hàng từ Oracle
query_cust = "SELECT customer_id, customer_name, age, gender FROM CUSTOMERS"
cust_lookup = pd.read_sql(query_cust, engine)

# 2. Chuẩn hóa tên cột về chữ thường để đồng bộ với code xử lý
cust_lookup.columns = cust_lookup.columns.str.lower()

# 3. (Tùy chọn) Làm sạch dữ liệu để merge chính xác hơn
cust_lookup['customer_name'] = cust_lookup['customer_name'].astype(str).str.strip()
cust_lookup['gender'] = cust_lookup['gender'].astype(str).str.strip()
cust_lookup['age'] = pd.to_numeric(cust_lookup['age'], errors='coerce').fillna(0).astype(int)
df_items_with_cus_id = pd.merge(
    df_orders_raw, 
    cust_lookup[['customer_id', 'customer_name', 'age', 'gender']], 
    on=['customer_name', 'age', 'gender'], 
    how='inner'
)
if 'customer_id_y' in df_items_with_cus_id.columns:
    df_items_with_cus_id = df_items_with_cus_id.rename(columns={'customer_id_y': 'customer_id'})

In [68]:
df_items_with_cus_id.drop_duplicates(['order_id', 'order_date', 'product_id'], inplace=True)

In [69]:
df_items_with_cus_id.info()

<class 'pandas.DataFrame'>
Index: 333375 entries, 0 to 418677
Data columns (total 26 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   order_id                  333375 non-null  str    
 1   order_date                333375 non-null  str    
 2   order_status              333375 non-null  str    
 3   customer_id_x             333375 non-null  str    
 4   customer_name             333375 non-null  str    
 5   gender                    333375 non-null  str    
 6   age                       333375 non-null  int64  
 7   customer_segment          333375 non-null  str    
 8   country                   333375 non-null  str    
 9   city                      333375 non-null  str    
 10  customer_loyalty_score    333375 non-null  float64
 11  total_orders_by_customer  333375 non-null  int64  
 12  product_id                333375 non-null  str    
 13  product_name              333375 non-null  str    
 14  cate

In [70]:
df_items_with_cus_id.head()


,order_id,order_date,order_status,customer_id_x,customer_name,gender,age,customer_segment,country,city,...,stock_quantity,unit_price_usd,quantity,discount_percent,discount_amount_usd,total_price_usd,tax_usd,payment_method,warehouse_location,customer_id
0,ORD-8VFLV,2025-09-24 22:44:05.262791,Completed,CUS-08WDW,Tony Medina,Male,38,Regular,Italy,Rome,...,391,449.68,5,0,0.00,2248.40,158.56,Debit Card,USA-East,665594
1,ORD-PKS0P,2024-06-03 12:16:05.367571,Completed,CUS-FLUVN,Madeline Morrison,Female,32,VIP,Australia,Perth,...,914,219.26,2,10,43.85,394.67,30.75,Debit Card,USA-West,665595
2,ORD-8LZJB,2025-03-02 06:36:11.910794,Completed,CUS-VBUIW,James Hunter,Male,72,Regular,France,Lyon,...,364,472.29,5,0,0.00,2361.45,181.87,Bank Transfer,USA-East,541841
4,ORD-QR558,2025-11-18 19:27:53.044985,Completed,CUS-IG10U,Heather Pruitt,Female,66,Regular,United States,New York,...,791,56.40,3,0,0.00,169.20,10.67,PayPal,UK,665597
5,ORD-CGL8K,2025-08-03 08:50:52.434477,Completed,CUS-P65RV,Paul Carroll,Male,51,VIP,Netherlands,Eindhoven,...,450,388.71,1,10,38.87,349.84,17.75,Debit Card,EU-Central,593341


In [71]:
df_items_with_cus_id['order_date'] = pd.to_datetime(df_items_with_cus_id['order_date']).dt.floor('s')
orders_db['order_date'] = pd.to_datetime(orders_db['order_date']).dt.floor('s')
df_ready = pd.merge(
    df_items_with_cus_id,
    orders_db[['order_id', 'customer_id', 'order_date']],
    on=['customer_id', 'order_date'],
    how='inner'
)

print(f"Số dòng khớp được ID mới: {len(df_ready)}")

Số dòng khớp được ID mới: 333375


In [72]:

if 'order_id_y' in df_ready.columns:
    df_ready = df_ready.rename(columns={'order_id_y': 'order_id'})

In [73]:
df_ready.info()

<class 'pandas.DataFrame'>
RangeIndex: 333375 entries, 0 to 333374
Data columns (total 27 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   order_id_x                333375 non-null  str           
 1   order_date                333375 non-null  datetime64[us]
 2   order_status              333375 non-null  str           
 3   customer_id_x             333375 non-null  str           
 4   customer_name             333375 non-null  str           
 5   gender                    333375 non-null  str           
 6   age                       333375 non-null  int64         
 7   customer_segment          333375 non-null  str           
 8   country                   333375 non-null  str           
 9   city                      333375 non-null  str           
 10  customer_loyalty_score    333375 non-null  float64       
 11  total_orders_by_customer  333375 non-null  int64         
 12  product_id   

In [74]:
df_ready.head()

,order_id_x,order_date,order_status,customer_id_x,customer_name,gender,age,customer_segment,country,city,...,unit_price_usd,quantity,discount_percent,discount_amount_usd,total_price_usd,tax_usd,payment_method,warehouse_location,customer_id,order_id
0,ORD-8VFLV,2025-09-24 22:44:05,Completed,CUS-08WDW,Tony Medina,Male,38,Regular,Italy,Rome,...,449.68,5,0,0.00,2248.40,158.56,Debit Card,USA-East,665594,940346
1,ORD-PKS0P,2024-06-03 12:16:05,Completed,CUS-FLUVN,Madeline Morrison,Female,32,VIP,Australia,Perth,...,219.26,2,10,43.85,394.67,30.75,Debit Card,USA-West,665595,722143
2,ORD-8LZJB,2025-03-02 06:36:11,Completed,CUS-VBUIW,James Hunter,Male,72,Regular,France,Lyon,...,472.29,5,0,0.00,2361.45,181.87,Bank Transfer,USA-East,541841,845737
3,ORD-QR558,2025-11-18 19:27:53,Completed,CUS-IG10U,Heather Pruitt,Female,66,Regular,United States,New York,...,56.40,3,0,0.00,169.20,10.67,PayPal,UK,665597,965388
4,ORD-CGL8K,2025-08-03 08:50:52,Completed,CUS-P65RV,Paul Carroll,Male,51,VIP,Netherlands,Eindhoven,...,388.71,1,10,38.87,349.84,17.75,Debit Card,EU-Central,593341,916294


In [75]:
import gc
import sys
import pandas as pd

print("=========================================================================")
print("📊 TIẾN TRÌNH XÓA CACHE VÀ GIẢI PHÓNG BỘ NHỚ RAM CHUYÊN SÂU")
print("=========================================================================")

# -------------------------------------------------------------------------
# BƯỚC 1: Chủ động xóa bỏ toàn bộ các DataFrame trung gian không còn sử dụng
# -------------------------------------------------------------------------
# Danh sách các biến đệm lớn tích tụ qua các bước xử lý trước đó
cac_bien_can_giai_phong = [
    'df_orders_raw', 'df_items', 'df_items_with_cus_id', 
    'df_orders_mapped', 'source_df', 'stock_df', 'products_clean'
]

count_del = 0
for ten_bien in cac_bien_can_giai_phong:
    # Kiểm tra chính xác sự tồn tại của biến trong bộ nhớ hệ thống trước khi xóa
    if ten_bien in globals() or ten_bien in locals():
        if ten_bien in globals():
            del globals()[ten_bien]
        else:
            del locals()[ten_bien]
        count_del += 1
        print(f"👉 Đã hủy liên kết và giải phóng biến dữ liệu: '{ten_bien}'")

if count_del == 0:
    print("-> Không có biến dữ liệu rác nào cần giải phóng.")

# -------------------------------------------------------------------------
# BƯỚC 2: Tẩy sạch bộ nhớ đệm lịch sử (Execution History Cache) của Jupyter
# -------------------------------------------------------------------------
# Đây là bước tối quan trọng để triệt tiêu các tham chiếu ẩn ngăn cản việc giải phóng RAM
try:
    # Xóa lịch sử danh sách lệnh đầu vào và đầu ra của IPython
    if 'In' in globals():
        globals()['In'] = []
    if 'Out' in globals():
        globals()['Out'].clear()
    
    # Xóa các biến đệm hệ thống tự sinh dạng _1, _2, _bước_chạy
    cac_bien_he_thong = list(globals().keys())
    for bien_an in cac_bien_he_thong:
        if bien_an.startswith('_') and bien_an[1:].isdigit():
            del globals()[bien_an]
            
    print("👉 Đã dọn sạch hoàn toàn bộ nhớ đệm lịch sử (In/Out Cache) của Notebook.")
except Exception as e:
    print(f"⚠️ Lưu ý trong quá trình xóa bộ nhớ đệm IPython: {e}")

# -------------------------------------------------------------------------
# BƯỚC 3: Triệu hồi Trình thu gom rác hệ thống (Garbage Collector)
# -------------------------------------------------------------------------
# Ép Python phải thu hồi và trả lại các phân vùng RAM trống cho Hệ điều hành ngay lập tức
gc.collect()
so_doi_tuong_da_huy = gc.collect()

print("-------------------------------------------------------------------------")
print(f"✅ HOÀN TẤT! Trình dọn rác đã giải phóng thành công {so_doi_tuong_da_huy} đối tượng.")
print("Hệ thống hiện tại đã hoàn toàn sạch sẽ, sẵn sàng chạy các Cell tiếp theo.")
print("=========================================================================")

📊 TIẾN TRÌNH XÓA CACHE VÀ GIẢI PHÓNG BỘ NHỚ RAM CHUYÊN SÂU
👉 Đã hủy liên kết và giải phóng biến dữ liệu: 'df_orders_raw'
👉 Đã hủy liên kết và giải phóng biến dữ liệu: 'df_items'
👉 Đã hủy liên kết và giải phóng biến dữ liệu: 'df_items_with_cus_id'
👉 Đã dọn sạch hoàn toàn bộ nhớ đệm lịch sử (In/Out Cache) của Notebook.
-------------------------------------------------------------------------
✅ HOÀN TẤT! Trình dọn rác đã giải phóng thành công 0 đối tượng.
Hệ thống hiện tại đã hoàn toàn sạch sẽ, sẵn sàng chạy các Cell tiếp theo.


In [76]:
import pandas as pd
import gc

# =========================================================================
# # 5. KẾT HỢP LẤY PRODUCT_ID (Cấu hình tối ưu hóa chống tràn RAM)
# =========================================================================

print(f"Số lượng dòng gốc của df_ready: {len(df_ready)}")

# Bước 1: Trích xuất và lọc trùng danh mục sản phẩm ngay tại chỗ để làm bảng tra cứu (Lookup Table)
# Chỉ giữ lại duy nhất 1 dòng đầu tiên tìm thấy cho mỗi tên sản phẩm
products_lookup = products_db[['product_id', 'product_name']].drop_duplicates(subset=['product_name'], keep='first')
print(f"Số lượng sản phẩm duy nhất sau khi lọc trùng danh mục: {len(products_lookup)}")

# Bước 2: Thực hiện phép toán Merge an toàn tuyệt đối (Mối quan hệ Nhiều - 1)
df_final_items = pd.merge(
    df_ready, 
    products_lookup, 
    on='product_name', 
    how='inner'
)

print(f"✅ THÀNH CÔNG! Số dòng của bảng kết quả df_final_items: {len(df_final_items)}")

# Bước 3: Chủ động giải phóng phân vùng bộ nhớ đệm tạm thời vừa tạo
del products_lookup
gc.collect()

Số lượng dòng gốc của df_ready: 333375
Số lượng sản phẩm duy nhất sau khi lọc trùng danh mục: 48
✅ THÀNH CÔNG! Số dòng của bảng kết quả df_final_items: 333375


0

In [77]:
if 'product_id_y' in df_final_items.columns:
    df_final_items = df_final_items.rename(columns={'product_id_y': 'product_id'})

In [78]:
df_final_items.drop_duplicates(['order_id', 'order_date', 'product_id_x'],inplace=True)

In [79]:
df_final_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 333375 entries, 0 to 333374
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   order_id_x                333375 non-null  str           
 1   order_date                333375 non-null  datetime64[us]
 2   order_status              333375 non-null  str           
 3   customer_id_x             333375 non-null  str           
 4   customer_name             333375 non-null  str           
 5   gender                    333375 non-null  str           
 6   age                       333375 non-null  int64         
 7   customer_segment          333375 non-null  str           
 8   country                   333375 non-null  str           
 9   city                      333375 non-null  str           
 10  customer_loyalty_score    333375 non-null  float64       
 11  total_orders_by_customer  333375 non-null  int64         
 12  product_id_x 

In [80]:
df_final_items.head()

,order_id_x,order_date,order_status,customer_id_x,customer_name,gender,age,customer_segment,country,city,...,quantity,discount_percent,discount_amount_usd,total_price_usd,tax_usd,payment_method,warehouse_location,customer_id,order_id,product_id
0,ORD-8VFLV,2025-09-24 22:44:05,Completed,CUS-08WDW,Tony Medina,Male,38,Regular,Italy,Rome,...,5,0,0.00,2248.40,158.56,Debit Card,USA-East,665594,940346,691
1,ORD-PKS0P,2024-06-03 12:16:05,Completed,CUS-FLUVN,Madeline Morrison,Female,32,VIP,Australia,Perth,...,2,10,43.85,394.67,30.75,Debit Card,USA-West,665595,722143,683
2,ORD-8LZJB,2025-03-02 06:36:11,Completed,CUS-VBUIW,James Hunter,Male,72,Regular,France,Lyon,...,5,0,0.00,2361.45,181.87,Bank Transfer,USA-East,541841,845737,675
3,ORD-QR558,2025-11-18 19:27:53,Completed,CUS-IG10U,Heather Pruitt,Female,66,Regular,United States,New York,...,3,0,0.00,169.20,10.67,PayPal,UK,665597,965388,779
4,ORD-CGL8K,2025-08-03 08:50:52,Completed,CUS-P65RV,Paul Carroll,Male,51,VIP,Netherlands,Eindhoven,...,1,10,38.87,349.84,17.75,Debit Card,EU-Central,593341,916294,744


In [81]:
# 3. Ép kiểu dữ liệu lần cuối để chắc chắn không còn "chuỗi" trong các cột số
# Oracle rất nghiêm ngặt với kiểu NUMBER
for col in ['order_id', 'product_id', 'quantity']:
    df_final_items[col] = pd.to_numeric(df_final_items[col], errors='coerce').astype(int)

for col in ['discount_percent', 'discount_amount_usd', 'unit_price_usd', 'total_price_usd']:
    df_final_items[col] = pd.to_numeric(df_final_items[col], errors='coerce').astype(float)


In [82]:
# 4. TÍNH TOÁN CÁC CỘT GIÁ TRỊ (Nếu CSV không có sẵn)
#df_final_items['unit_price_usd'] = df_final_items['unit_price_usd'] # Lấy giá từ DB
df_final_items['discount_amount_usd'] = df_final_items['quantity'] * df_final_items['unit_price_usd'] * (df_final_items['discount_percent'] / 100)
df_final_items['total_price_usd'] = (df_final_items['quantity'] * df_final_items['unit_price_usd']) - df_final_items['discount_amount_usd']

# 5. Lọc cột và nạp vào Oracle
cols_to_load = [
    'order_id', 'product_id', 'quantity', 
    'discount_percent', 'discount_amount_usd', 
    'unit_price_usd', 'total_price_usd'
]


In [83]:
df_final_items.drop_duplicates(['order_id', 'order_date', 'product_id_x'],inplace=True)

In [84]:
df_final_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 333375 entries, 0 to 333374
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   order_id_x                333375 non-null  str           
 1   order_date                333375 non-null  datetime64[us]
 2   order_status              333375 non-null  str           
 3   customer_id_x             333375 non-null  str           
 4   customer_name             333375 non-null  str           
 5   gender                    333375 non-null  str           
 6   age                       333375 non-null  int64         
 7   customer_segment          333375 non-null  str           
 8   country                   333375 non-null  str           
 9   city                      333375 non-null  str           
 10  customer_loyalty_score    333375 non-null  float64       
 11  total_orders_by_customer  333375 non-null  int64         
 12  product_id_x 

In [85]:
# 5. Nạp vào Oracle
try:
    print(f"Đang nạp {len(df_final_items)} dòng vào bảng ORDER_ITEMS...")
    df_final_items[cols_to_load].to_sql(
        'ORDER_ITEMS', 
        con=engine, 
        if_exists='append', 
        index=False, 
        chunksize=5000
    )
    print("✅ Nạp dữ liệu thành công!")
except Exception as e:
    print(f"❌ Lỗi khi nạp: {e}")

Đang nạp 333375 dòng vào bảng ORDER_ITEMS...
✅ Nạp dữ liệu thành công!


C:\Users\meiln\AppData\Local\Temp\ipykernel_4232\1835473429.py:4: UserWarning: The provided table name 'ORDER_ITEMS' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df_final_items[cols_to_load].to_sql(
